In [2]:
import pdfplumber
from pathlib import Path
from pdfplumber.utils.text import extract_text
import re
import json

In [3]:
ROOT = Path("~/code/selector_examenes").expanduser()
FORMULATION_PATH = ROOT / "Formulación"

In [4]:
file = ROOT / "Formulación/2020 - Formulación.pdf"

In [5]:
def extraer_nombres_formulas(statement):
    patron = re.compile(r"R\s*E\s*S\s*O\s*L\s*U\s*C\s*I\s*[ÓO]\s*N", re.IGNORECASE)
    m = patron.search(statement)
    end_position = m.start() if m else -1
    compounds_end_position = statement.lower().find("d)")

    # extraer nombre examen
    statement = statement[:end_position]
    lines = statement.strip().split("\n")
    exam_name = lines[-1].lower()
    exam_name = re.sub(r"ejercicio \w+", "", exam_name)

    # compuestos a nombrar
    compounds = statement[:compounds_end_position]
    compounds = compounds.replace("Formule o nombre los compuestos siguientes: ", "").strip()
    return exam_name, compounds

In [6]:
formulation_statements = {}
for file in FORMULATION_PATH.iterdir():
    if file.is_file():
        with pdfplumber.open(file) as pdf:
            year = int(file.name[:4])
            formulation_statements[year] = []
            for page in pdf.pages[1:]:
                statement = page.extract_text()
                formulation_statements[year].append(extraer_nombres_formulas(statement))

In [7]:
with open(ROOT / "formulation_statements.json", "w", encoding="utf-8") as f:
    json.dump(formulation_statements, f, ensure_ascii=False, indent=2)